# Experiment 03 — Wheeler-DeWitt Superspace Spawn

Validates **RBLE Eq. (8)** discrete momentum constraint:

$$\hat{H}\Psi = \sum_k \delta(p_k - \Phi_{\mathrm{stream}})$$

`StreamPacket` $\to$ `WDWGenerator.inject_stream` $\to$ $N$ `Wavepacket` objects with metric mutations $\delta g_k$.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "polomni").is_dir():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    nx = None

%matplotlib inline
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})
print(f"polomni root: {ROOT}")


## Construct parent unified state and stream packet


In [ ]:
from polomni.core.state.unified_state import UnifiedStateVector
from polomni.core.state.stream_packet import StreamPacket
from polomni.core.superspace.wdw_generator import WDWGenerator, Wavepacket

NUM_CHOICES = 5
phi = np.linspace(0.2, 1.0, NUM_CHOICES)
weights = np.exp(phi - phi.max())
weights = weights / weights.sum()

parent = UnifiedStateVector(
    X_spatial=np.array([0.0, 0.0, 0.0]),
    P_momentum=np.zeros(NUM_CHOICES),
    Lambda_laws=np.array([1.0e-120, 6.674e-11]),
    C_choice=weights.copy(),
    t=0.0,
)

packet = StreamPacket(
    district_id=1,
    parent_id=0,
    num_choices=NUM_CHOICES,
    phi_stream=phi.tolist(),
    branch_weights=weights.tolist(),
    information_trace=float(np.sum(phi**2)),
)
print(f"Parent dim={parent.dim}, packet branches={packet.num_choices}")


## WDWGenerator: spawn wavepackets


In [ ]:
gen = WDWGenerator(metric_mutation_scale=0.12)
wavepackets = gen.inject_stream(packet, parent)

assert isinstance(wavepackets, list)
assert len(wavepackets) == NUM_CHOICES
assert all(isinstance(wp, Wavepacket) for wp in wavepackets)
print(f"Spawned {len(wavepackets)} wavepackets")


## Verify spawn count equals num_choices


In [ ]:
assert len(wavepackets) == packet.num_choices == NUM_CHOICES

branch_ids = [wp.branch_id for wp in wavepackets]
assert branch_ids == list(range(NUM_CHOICES))

for wp in wavepackets:
    assert wp.metric_mutation.shape[0] == wp.metric_mutation.shape[1]
    assert wp.momentum_vector.size >= 1

print("Spawn count invariant verified.")


## Momentum delta-kick: residuals after injection


In [ ]:
residuals = np.array([wp.stream_residual for wp in wavepackets])
assert np.all(residuals >= 0.0)
print("stream_residual per branch:", np.round(residuals, 6))


## Plot wavepacket metric mutations (diagonal $\delta g_k$)


In [ ]:
mutations = np.array([np.trace(wp.metric_mutation) for wp in wavepackets])
momenta_norms = np.array([np.linalg.norm(wp.momentum_vector) for wp in wavepackets])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(NUM_CHOICES), mutations, "o-", color="#8e44ad", lw=2)
axes[0].set_xlabel("branch k")
axes[0].set_ylabel("Tr(delta g_k)")
axes[0].set_title("Metric mutation magnitude per branch")

axes[1].bar(np.arange(NUM_CHOICES), momenta_norms, color="#16a085")
axes[1].set_xlabel("branch k")
axes[1].set_ylabel("||p_k||")
axes[1].set_title("Momentum vector norms after injection")
plt.tight_layout()
plt.show()


## Heatmap of metric mutation matrices


In [ ]:
fig, axes = plt.subplots(1, NUM_CHOICES, figsize=(14, 3))
for k, (wp, ax) in enumerate(zip(wavepackets, axes)):
    im = ax.imshow(wp.metric_mutation, cmap="coolwarm")
    ax.set_title(f"k={k}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Metric mutation matrices delta g_k (RBLE Eq. 8)")
plt.tight_layout()
plt.show()


## Compare spawn_wavepackets vs inject_stream


In [ ]:
raw = gen.spawn_wavepackets(NUM_CHOICES, phi)
injected = gen.inject_stream(packet, parent)

assert len(raw) == len(injected) == NUM_CHOICES
# inject_stream should reduce residuals vs raw spawn
raw_res = np.array([wp.stream_residual for wp in raw])
inj_res = np.array([wp.stream_residual for wp in injected])
print("mean residual raw:", raw_res.mean())
print("mean residual injected:", inj_res.mean())


## BranchOperator split of unified state


In [ ]:
from polomni.core.superspace.branch_operator import BranchOperator

bo = BranchOperator()
splits = bo.split_state_vector(parent, NUM_CHOICES)
assert len(splits) == NUM_CHOICES
norms = [s.norm() for s in splits]
print("Split state norms:", np.round(norms, 4))


## Momentum alignment with Phi_stream


In [ ]:
phi_arr = np.asarray(packet.phi_stream)
alignments = []
for wp in wavepackets:
    p = np.asarray(wp.momentum_vector, dtype=float).ravel()
    n = min(p.size, phi_arr.size)
    cos = np.dot(p[:n], phi_arr[:n]) / (np.linalg.norm(p[:n]) * np.linalg.norm(phi_arr[:n]) + 1e-15)
    alignments.append(cos)

fig, ax = plt.subplots()
ax.bar(np.arange(NUM_CHOICES), alignments, color="#2980b9")
ax.set_ylim(0, 1.05)
ax.set_xlabel("branch k")
ax.set_ylabel("cos(p_k, Phi_stream)")
ax.set_title("Momentum alignment after delta-kick injection")
plt.tight_layout()
plt.show()


## Conclusions

1. `WDWGenerator.inject_stream` spawns exactly `num_choices` wavepackets.
2. Each wavepacket carries a positive-semidefinite metric mutation and momentum vector aligned with $\Phi_{\mathrm{stream}}$.
3. Stream residuals quantify deviation from the delta-kick constraint $\delta(p_k - \Phi_{\mathrm{stream}})$; injection minimizes these relative to raw spawn.
4. Metric mutations grow with branch index, consistent with the discrete WDW bifurcation rule (RBLE Eq. 8).
